In [1]:
import pandas as pd
import os
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from sentence_transformers import SentenceTransformer
import string
import re
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 100)

nlp = spacy.load("en_core_web_lg")

/opt/anaconda3/envs/nlp_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)  # This shows full column content
pd.set_option('display.width', None)  # Auto-detect display width
pd.set_option('display.max_seq_items', None)  # Show all items in lists

In [3]:
df = pd.read_csv('../data/dcInbox/dcinbox_export_116.csv')
unnamed_cols = df.columns.str.contains('^Unnamed')
df = df.loc[:, ~unnamed_cols].copy()
df = df[pd.to_numeric(df['Unix Timestamp'], errors='coerce').notna()].copy()
df['datetime'] = pd.to_datetime(df['Unix Timestamp'], unit='ms')

# Sort chronologically
df = df.sort_values('datetime').reset_index(drop=True)


/var/folders/02/c1hvrmj11kx0z457p84l6pbc0000gn/T/ipykernel_80140/3738974941.py:1: DtypeWarning: Columns (2,4,10,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197,198,199,200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224,225,226,227,228,229,230,231,232,233,234,235,236,237,238,239,240,241,242,243,244,245,246,247,248,249,250,251,252,253,254,255,256,25

In [4]:
# print min and max dates
print("Min date:", df['datetime'].min())
print("Max date:", df['datetime'].max())

Min date: 2019-01-03 10:06:40
Max date: 2021-01-02 23:26:40


In [5]:
## Preprocessing functions to apply to catalogs' aggregated course descriptions
def remove_double_spaces(text):
    if not text:
        return ""
    return " ".join(text.split())


def remove_paragraphs_dash(text):
    return re.sub(r"-\n", "", text)


def remove_paragraphs(text):
    return re.sub("\n", " ", text)

In [38]:
LEXICON = {    
    'Election Integrity Terms': [
        'election integrity',
        'voter fraud',
        'voter id',
        'election reform',
        'voter verification',
        'ballot security',
        'election security',
        'mail-in voting concerns',
        'voter suppression',
        'election audits',
        'American elections',
        'proof of citizenship',
        'election interference',
        'foreign interference',
        'paper ballots',
        'voting machines',
        'cybersecurity',
        'election theft',
    ],
}

In [7]:
content = (
    df['Body']
    .apply(remove_double_spaces)
    .apply(remove_paragraphs)
    .apply(remove_paragraphs_dash)
    .to_numpy()
)

In [8]:
model = SentenceTransformer("all-MiniLM-L6-v2")
X = model.encode(content, show_progress_bar=True)

Batches: 100%|██████████| 962/962 [03:57<00:00,  4.05it/s]


In [9]:
print(f"content.shape: {content.shape}")
print(f"X.shape: {X.shape}")


content.shape: (30778,)
X.shape: (30778, 384)


In [39]:
election_integrity_terms = LEXICON['Election Integrity Terms']
election_integrity_qv = model.encode(election_integrity_terms, show_progress_bar=True)
similarity_matrix = cosine_similarity(election_integrity_qv, X)

Batches: 100%|██████████| 1/1 [00:00<00:00,  3.00it/s]


In [36]:
threshold = 0.55

# Convert numpy array to DataFrame
similarity_matrix_df = pd.DataFrame(
    data=similarity_matrix.T,  # Transpose so rows=emails, columns=terms
    columns=election_integrity_terms
)
similarity_matrix_df['ID'] = df['ID'].values

# Find rows where ANY term has similarity > threshold
mask = (similarity_matrix_df[election_integrity_terms] > threshold).any(axis=1)
high_similarity_ids = similarity_matrix_df[mask]['ID'].values

# Create new dataframe with all original columns plus similarity info
filtered_df = df[df['ID'].isin(high_similarity_ids)].copy()

# Optionally, add columns showing which terms matched and their scores
for term in election_integrity_terms:
    # Merge the similarity scores back to the filtered dataframe
    term_scores = similarity_matrix_df[['ID', term]].rename(columns={term: f'{term}_score'})
    filtered_df = filtered_df.merge(term_scores, on='ID', how='left')

# Filter out score columns where the value is <= threshold (optional cleanup)
score_columns = [col for col in filtered_df.columns if col.endswith('_score')]
for col in score_columns:
    filtered_df.loc[filtered_df[col] <= threshold, col] = None

print(f"Found {len(filtered_df)} emails with similarity scores > {threshold}")
print(f"\nDataframe shape: {filtered_df.shape}")

Found 73 emails with similarity scores > 0.55

Dataframe shape: (73, 33)


In [37]:
filtered_df.to_csv('../data/election_integrity_sample_emails.csv')